<br>

# Resquests Post

<br>

Michel Metran\
Data: 04.11.2025\
Atualizado em: 04.11.2025


In [16]:
import pprint
from pathlib import Path
from urllib.parse import (parse_qs, parse_qsl, unquote, unquote_plus,
                          urlencode, urlsplit)

import requests

import pyFDBS
from pyFDBS.scraper.webdriver import Firefox

<br>

---

## Entendendo a URL

Inicialmente descobri que o método `POST` tem um _payload_ similar ao abaixo.


In [17]:
payload = "action=download&as=ACRELANDIA.tar&type=php-tar&baseHref=%2FAC%2F&hrefs=&hrefs%5B0%5D=%2FAC%2FACRELANDIA%2F"
payload = "action=download&as=ADOLFO.tar&type=php-tar&baseHref=%2FSP%2F&hrefs=&hrefs%5B0%5D=%2FSP%2FADOLFO%2F"

urlsplit(
    unquote(
        string=payload,
        encoding="utf-8",
    ),
)

SplitResult(scheme='', netloc='', path='action=download&as=ADOLFO.tar&type=php-tar&baseHref=/SP/&hrefs=&hrefs[0]=/SP/ADOLFO/', query='', fragment='')

<br>

A função `parse_qs` decompõe a _string_ e decodifica os valores (como `%2F` para `/`). O parâmetro `keep_blank_values=True` garante que `hrefs=` seja mantido. O parâmetro `encoding="utf-8"` é uma boa prática.


In [18]:
dados_decompostos = parse_qs(
    qs=payload,
    keep_blank_values=True,
    encoding="utf-8",
)
print(dados_decompostos)

{'action': ['download'], 'as': ['ADOLFO.tar'], 'type': ['php-tar'], 'baseHref': ['/SP/'], 'hrefs': [''], 'hrefs[0]': ['/SP/ADOLFO/']}


O melhor método que encontrei foi o `parse_qsl`.


In [19]:
params = dict(parse_qsl(payload))
params["hrefs"] = ""
params["action"] = '"download"'
params

{'action': '"download"',
 'as': 'ADOLFO.tar',
 'type': 'php-tar',
 'baseHref': '/SP/',
 'hrefs[0]': '/SP/ADOLFO/',
 'hrefs': ''}

Em linhas gerais temos um _json_. É esse arquivo que preciso abrir.


In [20]:
# dados = {
#     "action": '"download"',
#     "as": "ACRELANDIA.tar",
#     "type": "php-tar",
#     "baseHref": "/AC/",
#     "hrefs[0]": "/AC/ACRELANDIA/",
#     "hrefs": "",
# }

dados = {
    "action": "download",
    "as": "ADOLFO.tar",
    "type": "php-tar",
    "baseHref": "/SP/",
    "hrefs": "",
    "hrefs[0]": "/SP/ADOLFO/",
}

<br>

Como ele posso "encodar".


In [21]:
print(payload)
print(urlencode(params))
print(urlencode(dados))

action=download&as=ADOLFO.tar&type=php-tar&baseHref=%2FSP%2F&hrefs=&hrefs%5B0%5D=%2FSP%2FADOLFO%2F
action=%22download%22&as=ADOLFO.tar&type=php-tar&baseHref=%2FSP%2F&hrefs%5B0%5D=%2FSP%2FADOLFO%2F&hrefs=
action=download&as=ADOLFO.tar&type=php-tar&baseHref=%2FSP%2F&hrefs=&hrefs%5B0%5D=%2FSP%2FADOLFO%2F


In [22]:
print(payload == urlencode(params))
print(payload == urlencode(dados))

False
True


<br>

---

## Requisição


In [23]:
# 1. Defina seus Request Headers
headers = {
    #'Authorization': 'Bearer 12345abcdef',
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36",
    # Se usar o parâmetro 'json=payload', o requests define Content-Type
    "Content-Type": "application/x-www-form-urlencoded",
    # Se usar o parâmetro 'data=payload', o requests define Content-Type
    # Se passar string com 'data=', é bom definir o Content-Type:
    # 'Content-Type': 'application/json'
    #'Cookie': 'PHPSESSID=ce49648c5246824ee42b245661a412e5'
}

In [24]:
# create a requests session
session = requests.Session()

<br>

---

### _Cookies_

Eu achava que era necessário estar com os cookies na sessão. Em 18.08.2026 vi que não.


In [25]:
pprint.pprint(session.cookies)

<RequestsCookieJar[]>


In [26]:
# # Instancia Driver
# driver = Firefox(verify_ssl=False)
# driver.get(url="https://geo.fbds.org.br/SP/")

# # get the URL from Selenium
# # current_url = driver.current_url

# # add Selenium's cookies to requests
# selenium_cookies = driver.get_cookies()
# for cookie in selenium_cookies:
#     print(cookie)
#     # Adiciona Cookies na sessão
#     # session.cookies.set(name=cookie["name"], value=cookie["value"])

<br>

---

## Requisição


In [27]:
project_path = Path('.').absolute().parents[2]
print(project_path)

# Diretório de saída
output_dir = project_path / 'data' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)
output_dir

d:\Codes\GitHub\Org - Open Geodata\pyFBDS


WindowsPath('d:/Codes/GitHub/Org - Open Geodata/pyFBDS/data/output')

In [28]:
# Faz a requisição
r = session.post(
    url="https://geo.fbds.org.br/SP/?",
    # url="https://geo.fbds.org.br/",
    data=urlencode(dados),
    # Precisa ter o parâmetro headers
    headers=headers,
)
# Verifica se a requisição foi bem-sucedida (código 200, 201, 202, etc.)
r.raise_for_status()

# Salva Arquivo
with open(file=output_dir / "arquivo.tar", mode="wb") as f:
    f.write(r.content)

In [29]:
# Se a API retornar JSON, você pode acessar os dados da resposta:
dados_resposta = r.json()
dados_resposta

JSONDecodeError: Expecting value: line 1 column 1 (char 0)